# 08｜線性近似與泰勒多項式：用加減乘除算出 sin、eˣ、√

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/calculus-for-iih/blob/main/notebooks/08_taylor.ipynb)

臺北醫學大學 智慧醫療跨領域　微積分（1151）　**自由練習，不計分、不用繳交**

對應課堂投影片：**Calculus08 線性近似與泰勒多項式**

**怎麼用**
1. 按上方選單「執行階段 → 全部執行」，從上往下看圖和數字。
2. 看到 **✏️ 動手** 的格子，就自己改改看。填空沒填也不會出錯，程式會提醒你「還沒填」。
3. 卡住就點開「💡 提示」，真的不行再看「參考答案」。

不需要先會寫 Python，每一段程式前面都有中文說明。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial

def d(f, a, h=1e-5):
    "中央差分：用前後兩點估計導數（老師提供）。"
    return (f(a + h) - f(a - h)) / (2 * h)

def linear(f, a):
    "在 x = a 的線性近似 L(x) = f(a) + f′(a)(x − a)（老師提供）。"
    fa, slope = f(a), d(f, a)
    return lambda x: fa + slope * (x - a)

def check(ok, hint):
    """檢查答案：對了顯示 ✓，不對顯示提示。"""
    print("✓ 正確！" if ok else "✗ 還沒對，" + hint)

## 1. 放大再放大：曲線看起來像直線

$f(x)=\sqrt{x}$ 和它在 $x=4$ 的切線。三張圖的視窗一張比一張小，看看曲線和切線還分不分得開。

In [ ]:
f = np.sqrt
L = linear(f, 4)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
for a, w in zip(ax, [3, 0.5, 0.05]):          # 視窗半寬：3、0.5、0.05
    x = np.linspace(4 - w, 4 + w, 300)
    a.plot(x, f(x), color="#0B2E33", lw=3, label="√x")
    a.plot(x, L(x), "--", color="#D9534F", lw=2, label="tangent")
    a.set_title(f"x in [{4 - w:g}, {4 + w:g}]"); a.grid(alpha=0.3)
ax[0].legend(); plt.tight_layout(); plt.show()

for dx in [0.1, 1, 5]:
    print(f"離切點 {dx:>3}：√ = {f(4 + dx):.5f}，切線 = {L(4 + dx):.5f}，誤差 = {abs(f(4 + dx) - L(4 + dx)):.5f}")

**離切點越近，切線越準。** 這就是線性近似：$f(x)\approx f(a)+f'(a)(x-a)$。

## 2. ✏️ 動手 1：不用計算機估 √9.2

選一個好算的點 $a=9$：$f(9)=3$，$f'(x)=\dfrac{1}{2\sqrt{x}}$，所以 $f'(9)=\dfrac16$。

In [ ]:
answer_1 = None      # ✏️ 填空：√9.2 的線性近似（可以寫算式，例如 3 + 0.2 / 6）

# ---- 自動檢查（不用改）----
if answer_1 is None:
    print("還沒填，f(9) + f′(9) × 0.2。")
else:
    check(abs(answer_1 - (3 + 0.2 / 6)) < 1e-4, "f(9) = 3，斜率 1/6，往右走 0.2。")
    print(f"真正的 √9.2 = {np.sqrt(9.2):.6f}")

## 3. 高估還是低估？看 f″

$\sqrt{x}$ 是向下凹（$f''<0$），切線在曲線**上方**，所以線性近似會**高估**。$e^x$ 向上凹（$f''>0$），切線在下方，會**低估**。

In [ ]:
for name, g, a, x0 in [("√x", np.sqrt, 4, 4.5), ("eˣ", np.exp, 0, 0.3)]:
    Lg = linear(g, a)
    sign = "高估" if Lg(x0) > g(x0) else "低估"
    print(f"{name}：在 {a} 附近近似 {x0}，真值 {g(x0):.5f}，近似 {Lg(x0):.5f} → {sign}")

## 4. 微分 dy：體重多 2 kg，體表面積（BSA）變多少？

Mosteller 公式：$\text{BSA}=\sqrt{\dfrac{h\cdot w}{3600}}$（m²；h 身高 cm、w 體重 kg）。身高 170 cm 固定，體重 70 → 72 kg（示意）。

$d\text{BSA}=\dfrac{d\,\text{BSA}}{dw}\,dw$，和真正的變化量 $\Delta\text{BSA}$ 比一比。

In [ ]:
h_cm = 170
BSA = lambda w: np.sqrt(h_cm * w / 3600)
w0, dw = 70, 2

dBSA = d(BSA, w0) * dw                 # 微分（線性近似）
real = BSA(w0 + dw) - BSA(w0)          # 真正的變化
print(f"BSA(70) = {BSA(w0):.4f} m²")
print(f"dBSA ≈ {dBSA:.5f} m²，ΔBSA = {real:.5f} m²，差 {abs(dBSA - real):.6f}")

# 劑量若按 BSA 計算（示意：每 m² 給 100 mg）
print(f"按 100 mg/m² 計算，劑量大約多 {100 * dBSA:.1f} mg")

## 5. ✏️ 動手 2：相對誤差規則

腫瘤當成球：$V=\dfrac{\pi}{6}D^3$。量直徑 $D=2$ cm，可能誤差 0.1 cm（相對誤差 5%）。$y\propto x^n$ 時，**相對誤差乘以 n**。體積的相對誤差大約幾 %？

In [ ]:
answer_2 = None      # ✏️ 填空：體積相對誤差（%），只填數字

# ---- 自動檢查（不用改）----
V = lambda D: np.pi / 6 * D**3
if answer_2 is None:
    print("還沒填，V ∝ D³，n = 3。")
else:
    check(abs(answer_2 - 15) < 0.5, "5% × 3。")
    print(f"實際用 D = 2.1 算：體積多了 {100 * (V(2.1) / V(2) - 1):.2f}%（線性近似 15%，很接近）")

## 6. 不只一條直線：cos x 的泰勒多項式

先分清楚兩個名詞：
- **泰勒展開式**（泰勒級數）：一直加下去、永遠不停，在收斂範圍內**等於**原函數：$\cos x = 1-\dfrac{x^2}{2!}+\dfrac{x^4}{4!}-\dfrac{x^6}{6!}+\cdots$
- **泰勒多項式** $P_N(x)$：只取到 $N$ 次就停，是**近似**：$\cos x \approx P_N(x)$。電腦實際算的都是這個。

改 `N`（最高次數）：0、2、4、6、10，看多項式在多大的範圍貼住 cos x。

In [ ]:
N = 4           # ← 改這裡：最高次數（偶數）

def cos_taylor(x, N):
    return sum((-1)**k * x**(2 * k) / factorial(2 * k) for k in range(N // 2 + 1))

x = np.linspace(-7, 7, 600)
plt.figure(figsize=(7.5, 3.6))
plt.plot(x, np.cos(x), color="#0B2E33", lw=3, label="cos x")
plt.plot(x, cos_taylor(x, N), "--", color="#D9534F", lw=2, label=f"P{N}(x)")
plt.ylim(-2, 2); plt.grid(alpha=0.3); plt.legend(); plt.show()

for x0 in [0.5, 1, 2, 3]:
    print(f"x = {x0}：cos = {np.cos(x0):.6f}，P{N} = {cos_taylor(x0, N):.6f}，誤差 {abs(np.cos(x0) - cos_taylor(x0, N)):.2e}")

次數越高，貼住的範圍越寬；但離中心 0 越遠，總是越不準。

## 7. 只用加法算出 e

把 $e^x$ 的泰勒展開式代 $x=1$：$e=e^1=1+1+\dfrac1{2!}+\dfrac1{3!}+\dfrac1{4!}+\cdots$。加到第幾項就停，就是用第幾次的泰勒多項式。

In [ ]:
total = 0
for k in range(10):
    total += 1 / factorial(k)
    print(f"加到 1/{k}! ：{total:.10f}，誤差 {abs(np.e - total):.2e}")

加 6 項（到 $1/5!$）誤差就不到 0.002，加 8 項不到萬分之一。階乘長得很快，後面的項越來越小。

## 8. ✏️ 動手 3：計算機怎麼算 sin(0.5)？

7 次泰勒多項式：$\sin x \approx x-\dfrac{x^3}{3!}+\dfrac{x^5}{5!}-\dfrac{x^7}{7!}$。請填 $5!$ 等於多少。

In [ ]:
five_factorial = None      # ✏️ 填空：5! = 5 × 4 × 3 × 2 × 1

# ---- 自動檢查（不用改）----
if five_factorial is None:
    print("還沒填，5 × 4 × 3 × 2 × 1。")
else:
    check(five_factorial == 120, "5 × 4 × 3 × 2 × 1。")
    x0 = 0.5
    approx = x0 - x0**3 / 6 + x0**5 / five_factorial - x0**7 / 5040
    print(f"四項加總：{approx:.10f}")
    print(f"np.sin  ：{np.sin(x0):.10f}")

只用加減乘除，四項就對到小數點後 8 位左右。

## 9. 讓 SymPy 幫你展開

`sp.series(函數, x, 中心, 次數)` 會列出泰勒展開式的前幾項；`.removeO()` 去掉後面「還有更多項」的記號 $O(\cdot)$，剩下的就是泰勒多項式，可以拿來核對手算。

In [ ]:
import sympy as sp
X = sp.symbols("x")
for expr in [sp.exp(X), sp.sin(X), sp.cos(X), sp.log(1 + X), 1 / (1 - X), sp.sqrt(X)]:
    center = 4 if expr == sp.sqrt(X) else 0
    print(f"{str(expr):>10}（中心 {center}）≈ {sp.series(expr, X, center, 4).removeO()}")

最後一行是第 1 節 √x 在 4 的展開。SymPy 把 $2+\frac{x-4}{4}$ 化簡成 $\frac{x}{4}+1$，這兩項正是切線；後面的 $(x-4)^2$、$(x-4)^3$ 項負責修正彎曲。

## 10. 誤差表：離中心多遠、用幾項

用 $e^x$ 的 n 次泰勒多項式，看誤差怎麼隨距離和項數變化。

In [ ]:
def exp_taylor(x, n):
    return sum(x**k / factorial(k) for k in range(n + 1))

xs = [0.1, 0.5, 1, 2, 3]
print("  次數 n  " + "".join(f"  x = {x:<6}" for x in xs))
for nn in [1, 2, 4, 6]:
    print(f"   {nn:>2}     " + "".join(f"  {abs(np.exp(x) - exp_taylor(x, nn)):.1e} " for x in xs))

往右（離中心遠）誤差變大，往下（項數多）誤差變小。

## 11. 藥物排除：短時間內 $e^{-kt}\approx 1-kt$

一級排除，剩下的比例 $e^{-kt}$（示意：$k=0.2$ /小時）。短時間內「排掉的比例」≈ $kt$。看看什麼時候這個近似開始不準。

In [ ]:
k = 0.2
t = np.linspace(0, 10, 300)
plt.figure(figsize=(7, 3.5))
plt.plot(t, np.exp(-k * t), color="#0B2E33", lw=3, label="exp(-kt)")
plt.plot(t, 1 - k * t, "--", color="#D9534F", lw=2, label="1 - kt")
plt.axhline(0, color="gray", lw=1); plt.xlabel("t (hours)"); plt.ylabel("fraction remaining")
plt.grid(alpha=0.3); plt.legend(); plt.show()

for tt in [0.5, 1, 2, 5]:
    print(f"t = {tt} h：真正排掉 {1 - np.exp(-k * tt):.3f}，近似 kt = {k * tt:.3f}")

t = 5 小時時近似說「排掉 100%」，真正只排掉約 63%。**線性近似只適合短時間。**

## 12. ✏️ 動手 4：半小時排掉多少？

$k=0.2$ /小時，$t=0.5$ 小時。用線性近似，排掉的比例大約是多少？（填小數）

In [ ]:
answer_4 = None      # ✏️ 填空：排掉的比例 ≈ kt

# ---- 自動檢查（不用改）----
if answer_4 is None:
    print("還沒填，k × t。")
else:
    check(abs(answer_4 - 0.1) < 1e-6, "0.2 × 0.5。")
    print(f"真正排掉的比例：{1 - np.exp(-0.2 * 0.5):.4f}")

## 13. 超出收斂範圍：ln(1+x) 的泰勒展開式

$\ln(1+x)=x-\dfrac{x^2}{2}+\dfrac{x^3}{3}-\cdots$ 只在 $-1<x\le1$ 收斂。比較 $x=0.5$ 和 $x=1.5$ 的部分和。

In [ ]:
def ln_partial(x, n):
    return sum((-1)**(k + 1) * x**k / k for k in range(1, n + 1))

for x0 in [0.5, 1.5]:
    print(f"x = {x0}（真值 ln(1+x) = {np.log(1 + x0):.5f}）")
    for nn in [2, 5, 10, 20]:
        print(f"   加到第 {nn:>2} 項：{ln_partial(x0, nn):>12.5f}")

$x=0.5$ 越加越準；$x=1.5$ 越加越離譜。**多加項不一定更好，要看收斂範圍。**

## 14. 用程式檢查 AI 的答案

> AI 說：$\sin x\approx x-\dfrac{x^3}{3}$，所以 $\sin(0.5)\approx 0.5-\dfrac{0.125}{3}=0.4583$。

In [ ]:
print("AI 的版本  ：", 0.5 - 0.5**3 / 3)
print("x − x³/3!  ：", 0.5 - 0.5**3 / 6)
print("np.sin(0.5)：", np.sin(0.5))

AI 漏掉了階乘：第三項的分母是 $3!=6$，不是 3。用數值比一比，馬上就看得出來。

## 小結

- 線性近似 $f(x)\approx f(a)+f'(a)(x-a)$：離切點越近越準；$f''$ 的正負決定高估或低估。
- 微分 $dy=f'(x)\,dx$ 估計變化量；$y\propto x^n$ 時相對誤差乘以 n。
- 泰勒多項式：$f(x)\approx\sum_{k=0}^{n} \dfrac{f^{(k)}(a)}{k!}(x-a)^k$（取到 n 次，近似），項數越多、離中心越近越準。
- 泰勒展開式（泰勒級數）：同一個式子一直加到無限多項，在收斂範圍內等於 $f(x)$。
- 超出收斂範圍，加再多項都沒用。

本週內容會考在期末考。

---
*本 notebook 為教學練習，數值與情境皆為示意，資料為程式產生的虛擬資料，非真實病人資料，也不是用藥建議。*